# EDA — Picoclimate track-based dataset
This notebook performs an exploratory data analysis (EDA) of the track-based CSV at `data/picoclimate_test/tracks_measurements.csv`.
Goals: load a sample, inspect schema, check missingness, plot track-length distribution, and sample time-series for a representative track.

In [4]:
# Imports
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")
%matplotlib inline

In [ ]:
# Paths and quick checks
# DATA_PATH = Path('data/picoclimate_test/tracks_measurements.csv')
DATA_PATH = Path(r"D:\repositories\personal\xai-spatio-temporal\data\picoclimate_test\tracks_measurements.csv")
print('Data path:', DATA_PATH)
print('Exists:', DATA_PATH.exists())
if DATA_PATH.exists():
    print('Size (MB):', DATA_PATH.stat().st_size / (1024**2))

Data path: D:\repositories\personal\xai-spatio-temporal\data\picoclimate_test\tracks_measurements.csv
Exists: True
Size (MB): 98.91149711608887


In [24]:
# Load a small sample first to inspect columns and dtypes
if DATA_PATH.exists():
    sample = pd.read_csv(DATA_PATH, nrows=1000, low_memory=False)
    display(sample.head())
    print('\nColumns:', list(sample.columns))
else:
    print('File not found — please ensure the CSV exists locally at', DATA_PATH)

,track_id,city,date,time_slot,slot_index,loc_id,loc_index,timestamp,true_regime,air_temp_c,...,co2_ppm,no2_ppb,o3_ppb,noise_db,traffic_index,pedestrian_index,sky_view_factor,impervious_fraction,water_proximity,heat_index_c
0,nan_track_000,Nantes,2026-05-02,noon,1,nan_track_000_loc_000,0,2026-05-02T12:00:00+00:00,dry_spell,13.183428,...,396.667112,21.460434,52.815136,48.902536,0.311897,0.202295,0.504989,0.554530,0.196857,14.404075
1,nan_track_000,Nantes,2026-05-02,noon,1,nan_track_000_loc_001,1,2026-05-02T12:10:00+00:00,dry_spell,16.147433,...,384.000310,14.314304,49.748408,49.351010,0.406288,0.073647,0.376633,0.530562,0.175490,17.319226
2,nan_track_000,Nantes,2026-05-02,noon,1,nan_track_000_loc_002,2,2026-05-02T12:20:00+00:00,dry_spell,14.002191,...,391.096997,19.708189,53.501103,45.003305,0.324462,0.094696,0.485197,0.530214,0.083219,15.671313
3,nan_track_000,Nantes,2026-05-02,noon,1,nan_track_000_loc_003,3,2026-05-02T12:30:00+00:00,dry_spell,16.357433,...,415.759597,10.698806,34.389410,43.349257,0.166813,0.208074,0.347858,0.552415,0.185028,18.027879
4,nan_track_000,Nantes,2026-05-02,noon,1,nan_track_000_loc_004,4,2026-05-02T12:40:00+00:00,dry_spell,14.147810,...,396.447340,34.700834,47.501256,47.412261,0.210641,0.225984,0.525087,0.515828,0.000000,15.234473



Columns: ['track_id', 'city', 'date', 'time_slot', 'slot_index', 'loc_id', 'loc_index', 'timestamp', 'true_regime', 'air_temp_c', 'rel_humidity_pct', 'wind_speed_ms', 'wind_dir_deg', 'pressure_hpa', 'precipitation_mm', 'solar_wm2', 'longwave_wm2', 'surface_temp_c', 'soil_moisture_pct', 'ndvi', 'pm25_ugm3', 'pm10_ugm3', 'co2_ppm', 'no2_ppb', 'o3_ppb', 'noise_db', 'traffic_index', 'pedestrian_index', 'sky_view_factor', 'impervious_fraction', 'water_proximity', 'heat_index_c']


In [7]:
# Load full file (careful: may be large). Use chunks if needed.
def load_df(path, nrows=None):
    if nrows is not None:
        return pd.read_csv(path, nrows=nrows, low_memory=False)
    return pd.read_csv(path, low_memory=False)

df = None
if DATA_PATH.exists():
    try:
        df = load_df(DATA_PATH)
    except MemoryError:
        print('MemoryError: try loading a smaller sample with nrows or use chunksize')

if df is not None:
    print('Loaded df shape:', df.shape)
    display(df.head())

In [17]:
df.head()

AttributeError: 'NoneType' object has no attribute 'head'

In [8]:
# Basic info and descriptive stats
if df is not None:
    display(df.info())
    display(df.describe(include='all'))

In [9]:
# Missing values summary
if df is not None:
    mv = df.isnull().sum().sort_values(ascending=False)
    display(mv[mv>0].head(50))

In [10]:
# Identify ID and time columns heuristically
id_candidates = ['loc_id','track_id','id','location_id','sensor_id']
time_candidates = ['timestamp','time','datetime','date','ts']
id_col = None
time_col = None
if df is not None:
    cols = set(df.columns.str.lower())
    for c in id_candidates:
        if c in cols:
            # find actual column name with that lowercase
            id_col = [x for x in df.columns if x.lower()==c][0]
            break
    for c in time_candidates:
        if c in cols:
            time_col = [x for x in df.columns if x.lower()==c][0]
            break
    print('Detected id_col ->', id_col, 'time_col ->', time_col)

In [11]:
# Track-length distribution (number of rows per id)
if df is not None and id_col is not None:
    track_counts = df.groupby(id_col).size().rename('count')
    display(track_counts.describe())
    plt.figure(figsize=(8,4))
    sns.histplot(x=track_counts.to_numpy(), bins=50, log_scale=(False, True))
    plt.title('Track length distribution (rows per track)')
    plt.xlabel('Rows per track')
    plt.tight_layout()
else:
    print('Cannot compute track lengths: id_col not detected or df missing')

Cannot compute track lengths: id_col not detected or df missing


In [16]:
# Numeric columns overview and distributions for top numeric features
if df is not None:
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    print('Numeric columns count:', len(num_cols))
    top_num = num_cols[:6]
    print('Top numeric columns:', top_num)
    if top_num:
        plt.figure(figsize=(12,8))
        for i, c in enumerate(top_num, 1):
            plt.subplot(2,3,i)
            sns.histplot(df[c].dropna(), bins=50)
            plt.title(c)
        plt.tight_layout()

In [13]:
# Sample time-series for a representative track
if df is not None and id_col is not None:
    top_id = track_counts.sort_values(ascending=False).index[0]
    print('Top id (most rows):', top_id)
    track_df = df[df[id_col]==top_id].copy()
    if time_col is not None:
        track_df[time_col] = pd.to_datetime(track_df[time_col], errors='coerce')
        track_df = track_df.sort_values(time_col)
    numeric = track_df.select_dtypes(include=[np.number]).columns.tolist()[:4]
    plt.figure(figsize=(12,6))
    for i,c in enumerate(numeric,1):
        plt.subplot(len(numeric),1,i)
        plt.plot(track_df[c].values)
        plt.title(f'{c} over time for {id_col}={top_id}')
    plt.tight_layout()
else:
    print('Cannot plot time-series: missing df or id_col')

Cannot plot time-series: missing df or id_col


In [15]:
# Save a compact summary to outputs/
OUT_DIR = Path('outputs/eda_picoclimate')
OUT_DIR.mkdir(parents=True, exist_ok=True)
if df is not None:
    summary = {
        'rows': int(df.shape[0]),
        'cols': int(df.shape[1]),
        'unique_ids': int(df[id_col].nunique()) if id_col is not None else None,
    }
    pd.DataFrame([summary]).to_csv(OUT_DIR / 'summary.csv', index=False)
    print('Wrote', OUT_DIR / 'summary.csv')

## Next steps
- Consider loading with `chunksize` if memory is constrained.
- Add specialized per-variable plots and seasonal/time-of-day aggregations.
- Export cleaned subset for modeling or shapelet extraction.